In [ ]:
q="""
  with byob_confirmed as (
    select distinct
      min(yyyymmdd) over(partition by eventprops__eventprops__userid order by yyyymmdd) as yyyymmdd,
      eventprops__eventprops__userid as captain_id
    from
      canonical.iceberg_production_data_raw_clevertap_captain_clevertap_captain_events_v2_eventname_byob_screen_immutable
    where eventprops__eventprops__action='plan_confirmed'
    and yyyymmdd between '20250124' and '20250214'
    and lower(eventprops__eventprops__city) in ('hyderabad','kolkata')
),
appo as(
select captain_id,gig_segment,row_number() over(partition by captain_id order by run_date desc) as r
from reports_internal.captain_gig_appography_v4
where city in ('hyderabad','kolkata')
),
finalAppo as( 
  select captain_id,gig_segment
 from appo
 where r=1
),
byob_test_caps as(
  select a.yyyymmdd,a.captain_id,b.performance_segment,b.consistency_segment,
      yyyymmdd as post_start_date,
      date_format(date_add('day',date_diff('day',date_parse(yyyymmdd,'%Y%m%d'),date_parse('20250224','%Y%m%d')),date_parse(yyyymmdd,'%Y%m%d')),'%Y%m%d') as post_end_date,
      date_format(date_add('day',-1,date_parse(yyyymmdd,'%Y%m%d')),'%Y%m%d') as pre_end_date,
      date_format(date_add('day',-(date_diff('day',date_parse(yyyymmdd,'%Y%m%d'),date_parse('20250224','%Y%m%d'))+1),date_parse(yyyymmdd,'%Y%m%d')),'%Y%m%d') as pre_start_date,
      c.gig_segment
  from byob_confirmed a
  left join (select captain_id,performance_segment,consistency_segment
    from(
        select distinct captain_id,performance_segment,consistency_segment
        from  hive.experiments_internal.captain_byob_experiment_2_base_test 
    )) b
  on a.captain_id=b.captain_id 
  left join finalAppo c
  on a.captain_id=c.captain_id 
),
orders as(
 select yyyymmdd,captain_id,count(distinct order_id) as numOrders
 from orders.order_logs_fact
 where order_status='dropped'
 and yyyymmdd between '20241223' and '20250224'
 and captain_id in (select captain_id from byob_test_caps)
 group by 1,2
),
cm as (
  select yyyymmdd,captain_id,sum(total_login_hours) as login_hrs
  from datasets.captain_metrics as cm
  where yyyymmdd between '20241223' and '20250224'
  and captain_id in (select captain_id from byob_test_caps)
  group by 1,2
),
totalEarning as( select yyyymmdd,rider_id as captain_id,
    coalesce(sum(case when service = 'Link' then captain_earnings end),0) as earning
    from captain.captain_transactions_snapshot
            where 1=1
            and (transaction_type = 'orders' or transaction_type='incentives')
            and yyyymmdd between '20250118' and '20250219'
            and rider_id in (select captain_id from byob_test_caps)
    group by 1,2
),
metric_orders as (
   select a.yyyymmdd,a.captain_id,a.numOrders,b.login_hrs
   from orders a
   inner join cm b
   on a.captain_id=b.captain_id
   and a.yyyymmdd=b.yyyymmdd
   where a.numOrders>0
),
metric_order_earning as(
  select a.yyyymmdd,a.captain_id,a.numOrders,a.login_hrs,b.earning
   from metric_orders a
   left join totalEarning b
   on a.captain_id=b.captain_id
   and a.yyyymmdd=b.yyyymmdd
),
finalTbl as (
   select a.yyyymmdd,
        count(distinct a.captain_id) as totalCaptains,
        count(case when b.yyyymmdd between a.pre_start_date and a.pre_end_date then b.yyyymmdd end) as pre_netDays,
        sum(case when b.yyyymmdd between a.pre_start_date and a.pre_end_date then b.numOrders else 0 end) as pre_numOrders,
        sum(case when b.yyyymmdd between a.pre_start_date and a.pre_end_date then b.login_hrs else 0 end) as pre_login_hrs,
        sum(case when b.yyyymmdd between a.pre_start_date and a.pre_end_date then b.earning else 0 end) as pre_total_earnings,
        count(distinct case when b.yyyymmdd between a.post_start_date and a.post_end_date then b.yyyymmdd end) as post_netDays,
        sum(case when b.yyyymmdd between a.post_start_date and a.post_end_date then b.numOrders else 0  end) as post_numOrders,
        sum(case when b.yyyymmdd between a.post_start_date and a.post_end_date then b.login_hrs else 0 end) as post_login_hrs,
        sum(case when b.yyyymmdd between a.post_start_date and a.post_end_date then b.earning else 0 end) as post_total_earnings
    from byob_test_caps a
    inner join metric_order_earning b
    on a.captain_id=b.captain_id
    group by 1
),
analysisTbl as (
    select yyyymmdd,totalCaptains,sum(pre_numOrders) as preOrders,sum(pre_netDays) as preNetDays,sum(pre_login_hrs) as preLoginHrs,
    sum(pre_total_earnings) as preTotalEarnings,
    sum(post_total_earnings) as postTotalEarnings,
    sum(post_numOrders) as postOrders,sum(post_netDays) as postNetDays,
    sum(post_login_hrs) as postLoginHrs
    from finalTbl
    group by 1
)
select * , (RPR_Post - RPR_Pre) as RPR_Change, (Net_days_Post - Net_days_Pre) as Net_days_Change, (LH_Post - LH_Pre) as LH_Change,(Earnings_post - Earnings_pre) as Earnings_change
from (
select yyyymmdd,
totalCaptains as Total_Captains,
(preOrders*1.0/totalCaptains) as RPR_Pre,  (preNetDays*1.0/totalCaptains) as Net_days_Pre, (preLoginHrs*1.0 / totalCaptains) as LH_Pre,(preTotalEarnings*1.0/totalCaptains) as Earnings_pre,
(postOrders*1.0/totalCaptains) as RPR_Post,  (postNetDays*1.0/totalCaptains) as Net_days_Post, (postLoginHrs*1.0 / totalCaptains) as LH_Post,(postTotalEarnings*1.0/totalCaptains) as Earnings_post
from analysisTbl
where totalCaptains>10
)
"""